# Alpamayo-Edge 一次性导出/量化 (Kaggle)

**运行前(右侧 Settings):** ① Accelerator = **GPU T4 x2** ② **Internet = On**
③ Add-ons → **Secrets** 新增 `HF_TOKEN`=你的 HF read token(已在 HF 网站同意 Alpamayo-R1 / Cosmos-Reason2 许可)。

产出 `/kaggle/working/edge_artifacts.tgz`(ONNX,几 GB)—— 跑完在右侧 Output 下载,再 scp 到 Orin。

> 显存提醒:单张 T4=16GB。Cosmos-8B 量化用 offload 可行;**Alpamayo-10B FP16 导出**可能 OOM,
> 脚本已设 `device_map=auto` 尽量分摊到两卡+CPU;真不行就先只交 Cosmos。


## 1. 环境检查 + HF token


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN']=UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded from Kaggle Secrets')
except Exception as e:
    os.environ['HF_TOKEN']='hf_粘贴你的Token'   # 若没设 Secret 就改这里
    print('using inline token')


## 2. 安装 TensorRT-Edge-LLM (Kaggle 直连 GitHub)


In [ ]:
!git clone --depth 1 https://github.com/NVIDIA/TensorRT-Edge-LLM.git /kaggle/working/TRTEdge
!cd /kaggle/working/TRTEdge && pip -q install ".[tools]"
import os; os.environ['PYTHONPATH']='/kaggle/working/TRTEdge'


## 3. Cosmos-Reason2-8B → INT4 / INT8 (先跑,验证链路)


In [ ]:
%cd /kaggle/working
import os
for QF in ['int4_awq','int8_sq']:
    rc=os.system(f'tensorrt-edgellm-quantize llm --model_dir nvidia/Cosmos-Reason2-8B --output_dir Cosmos-8B-{QF} --qformat {QF}')
    print('quantize',QF,'rc=',rc)
    os.system(f'tensorrt-edgellm-export Cosmos-8B-{QF} Cosmos-8B-{QF}/onnx')
print('cosmos done')


## 4. Alpamayo-R1-10B → FP16 ONNX (显存吃紧,OOM 就跳过只交 Cosmos)


In [ ]:
%cd /kaggle/working
!hf download nvidia/Alpamayo-R1-10B --local-dir Alpamayo-R1-10B
!tensorrt-edgellm-export Alpamayo-R1-10B Alpamayo-R1-10B/onnx --max-kv-cache-capacity 4096


## 5. 打包 (在右侧 Output 面板下载 edge_artifacts.tgz)


In [ ]:
%cd /kaggle/working
import glob,os
paths=[p for p in ['Alpamayo-R1-10B/onnx','Cosmos-8B-int4_awq/onnx','Cosmos-8B-int8_sq/onnx'] if os.path.isdir(p)]
os.system('tar -czf edge_artifacts.tgz '+' '.join(paths))
print('packed:',paths); print('size:',os.path.getsize('edge_artifacts.tgz')/1e6,'MB')
